<a href="https://colab.research.google.com/github/abhaysachan007/Fault-Detection-Challenge/blob/main/Fault_Detection_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
test_df = pd.read_csv("TEST.csv")
print("TEST rows:", len(test_df))

TEST rows: 10944


In [3]:
!pip -q install catboost xgboost lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.7 MB/s eta 0:00:00


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings

warnings.filterwarnings('ignore')

train = pd.read_csv("TRAIN.csv")
test = pd.read_csv("TEST.csv")

train = train.dropna(subset=["Class"]).reset_index(drop=True)
train["Class"] = train["Class"].astype(int)

features = [c for c in test.columns if c != "ID"]
X = train[features]
y = train["Class"].values
X_test = test[features]
test_ids = test["ID"].values

neg, pos = np.bincount(y)
scale_pos_weight = neg / pos

N_FOLDS = 5
kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_xgb = np.zeros(len(train))
oof_lgb = np.zeros(len(train))
oof_cat = np.zeros(len(train))

test_xgb = np.zeros(len(test))
test_lgb = np.zeros(len(test))
test_cat = np.zeros(len(test))

print(f"Starting training on {len(X)} rows with {N_FOLDS} folds...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
    X_tr, y_tr = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]

    xgb_model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        tree_method='hist',
        n_jobs=-1,
        random_state=42
    )
    xgb_model.fit(X_tr, y_tr)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_val)[:, 1]
    test_xgb += xgb_model.predict_proba(X_test)[:, 1] / N_FOLDS

    lgb_model = lgb.LGBMClassifier(
        n_estimators=1200,
        learning_rate=0.05,
        num_leaves=35,
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1,
        random_state=42,
        verbose=-1
    )
    lgb_model.fit(X_tr, y_tr)
    oof_lgb[val_idx] = lgb_model.predict_proba(X_val)[:, 1]
    test_lgb += lgb_model.predict_proba(X_test)[:, 1] / N_FOLDS

    cat_model = CatBoostClassifier(
        iterations=1200,
        learning_rate=0.05,
        depth=7,
        class_weights=[1, scale_pos_weight],
        verbose=0,
        random_seed=42
    )
    cat_model.fit(X_tr, y_tr)
    oof_cat[val_idx] = cat_model.predict_proba(X_val)[:, 1]
    test_cat += cat_model.predict_proba(X_test)[:, 1] / N_FOLDS

    print(f"Fold {fold} complete.")

X_meta = np.column_stack((oof_xgb, oof_lgb, oof_cat))
X_test_meta = np.column_stack((test_xgb, test_lgb, test_cat))

meta_model = LogisticRegression(random_state=42, C=0.3, solver="lbfgs", max_iter=2000)
meta_model.fit(X_meta, y)

oof_final = meta_model.predict_proba(X_meta)[:, 1]
test_probs_final = meta_model.predict_proba(X_test_meta)[:, 1]

thresholds = np.linspace(0.1, 0.9, 100)
best_threshold = 0.5
best_f1 = 0.0

for t in thresholds:
    preds = (oof_final > t).astype(int)
    score = f1_score(y, preds)
    if score > best_f1:
        best_f1 = score
        best_threshold = t

fine_low = max(0.05, best_threshold - 0.08)
fine_high = min(0.95, best_threshold + 0.08)
fine_thresholds = np.linspace(fine_low, fine_high, 321)

for t in fine_thresholds:
    preds = (oof_final > t).astype(int)
    score = f1_score(y, preds)
    if score > best_f1:
        best_f1 = score
        best_threshold = t

print(f"\nBest Threshold: {best_threshold:.4f}")
print(f"CV F1 Score: {best_f1:.5f}")

final_preds = (test_probs_final > best_threshold).astype(int)

submission = pd.DataFrame({
    "ID": test_ids,
    "CLASS": final_preds
})

submission.to_csv("FINAL.csv", index=False)
print(f"Saved FINAL.csv with {len(submission)} rows.")

Starting training on 43776 rows with 5 folds...
Fold 1 complete.
Fold 2 complete.
Fold 3 complete.
Fold 4 complete.
Fold 5 complete.

Best Threshold: 0.3586
CV F1 Score: 0.98597
Saved FINAL.csv with 10944 rows.


In [8]:
t = pd.read_csv("TEST.csv")
s = pd.read_csv("FINAL.csv")
print(len(t), len(s), list(s.columns), s["ID"].equals(t["ID"]), set(s["CLASS"].unique()))

10944 10944 ['ID', 'CLASS'] True {np.int64(0), np.int64(1)}
